<a href="https://colab.research.google.com/github/gez2code/dermamnist-hybrid-study/blob/main/Binary_Classification_DermaMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# BLOCK 1: INSTALLATION & IMPORTS
# ============================================================================
!pip install medmnist wandb

import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Deep Learning Imports
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K

# Metrics & Data
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils import class_weight
from medmnist import DermaMNIST

# Tracking
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

# Reproducibility
SEED = 42
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds()
print(f"✓ Setup complete. TensorFlow Version: {tf.__version__}")

✓ Setup complete. TensorFlow Version: 2.19.0


In [ ]:
# ============================================================================
# BLOCK 2: DATA LOADING & PREPROCESSING (UPDATED)
# ============================================================================
def load_and_preprocess_data():
    print("\nLoading DermaMNIST...")
    train_data = DermaMNIST(split='train', download=True, size=28)
    val_data = DermaMNIST(split='val', download=True, size=28)
    test_data = DermaMNIST(split='test', download=True, size=28)

    # Normalize
    x_train = train_data.imgs.astype('float32') / 255.0
    x_val = val_data.imgs.astype('float32') / 255.0
    x_test = test_data.imgs.astype('float32') / 255.0

    # Binary mapping: Malignant (1) vs Benign (0)
    to_binary = lambda y: np.isin(y, [0, 1, 6]).astype(int)
    y_train_bin = to_binary(train_data.labels)
    y_val_bin = to_binary(val_data.labels)
    y_test_bin = to_binary(test_data.labels)

    # One-hot encode
    y_train = tf.keras.utils.to_categorical(y_train_bin, 2)
    y_val = tf.keras.utils.to_categorical(y_val_bin, 2)
    y_test = tf.keras.utils.to_categorical(y_test_bin, 2)

    # UPDATED: NO class weights (following Yang et al. 2023 DermaMNIST approach)
    # Let the model learn naturally with augmentation
    class_weights = None  # Changed from {0: 1.0, 1: 3.0}

    print(f'✓ Data loaded.')
    print(f'  Train: {x_train.shape}, Malignant: {100*y_train_bin.mean():.1f}%')
    print(f'  Val:   {x_val.shape}, Malignant: {100*y_val_bin.mean():.1f}%')
    print(f'  Test:  {x_test.shape}, Malignant: {100*y_test_bin.mean():.1f}%')
    print(f'  Class weights: None (balanced training like literature)')

    return {
        'x_train': x_train, 'y_train': y_train, 'y_train_bin': y_train_bin,
        'x_val': x_val, 'y_val': y_val, 'y_val_bin': y_val_bin,
        'x_test': x_test, 'y_test': y_test, 'y_test_bin': y_test_bin,
        'class_weights': class_weights
    }

def get_augmentation_generator():
    return ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=0.1,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest'
    )

# Load Data Once
data = load_and_preprocess_data()
datagen = get_augmentation_generator()

# ============================================================================
# GOOGLE DRIVE VERBINDEN & SPEICHER-FUNKTION
# ============================================================================
from google.colab import drive
import shutil
import os

# 1. Drive mounten
drive.mount('/content/drive')

# 2. Ordner erstellen
SAVE_DIR = '/content/drive/MyDrive/DermaModels_Phase1'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Speicherort: {SAVE_DIR}")

# 3. Hilfsfunktion zum Speichern
def save_to_drive(experiment_name):
    source = f"{experiment_name}.keras"
    destination = os.path.join(SAVE_DIR, source)

    if os.path.exists(source):
        shutil.copy(source, destination)
        print(f"💾 GESPEICHERT: {experiment_name} -> Google Drive")
    else:
        print(f"⚠️ FEHLER: Modelldatei {source} nicht gefunden!")


Loading DermaMNIST...
✓ Data loaded.
  Train: (7007, 28, 28, 3), Malignant: 9.8%
  Val:   (1003, 28, 28, 3), Malignant: 9.9%
  Test:  (2005, 28, 28, 3), Malignant: 9.9%
  Class weights: None (balanced training like literature)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Speicherort: /content/drive/MyDrive/DermaModels_Phase1


In [ ]:
# ============================================================================
# MASTER SETUP: GOOGLE DRIVE & ORDNER-STRUKTUR
# ============================================================================
from google.colab import drive
import shutil
import os

# 1. Drive mounten
drive.mount('/content/drive')

# 2. Zentrale Definition der Ordner-Pfade
BASE_PATH = '/content/drive/MyDrive/DermaMNIST_Study'

# Wir definieren Variablen für jede Phase
DIR_PHASE_1 = os.path.join(BASE_PATH, 'Phase1_Baselines')
DIR_PHASE_2 = os.path.join(BASE_PATH, 'Phase2_Tuning')
DIR_PHASE_3 = os.path.join(BASE_PATH, 'Phase3_Final')

# Alle Ordner auf einmal erstellen
for folder in [DIR_PHASE_1, DIR_PHASE_2, DIR_PHASE_3]:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Ordner-Struktur erstellt unter: {BASE_PATH}")
print(f"   📂 Phase 1 -> {DIR_PHASE_1}")
print(f"   📂 Phase 2 -> {DIR_PHASE_2}")
print(f"   📂 Phase 3 -> {DIR_PHASE_3}")

# 3. Verbesserte Speicher-Funktion (akzeptiert Zielordner!)
def save_to_drive(experiment_name, target_folder):
    """
    Speichert das Modell in den spezifischen Phasen-Ordner.
    """
    filename = f"{experiment_name}.keras"
    source = filename # Liegt lokal im Colab Root
    destination = os.path.join(target_folder, filename)

    if os.path.exists(source):
        shutil.copy(source, destination)
        print(f"💾 GESPEICHERT: {filename} \n   -> {destination}")
    else:
        print(f"⚠️ FEHLER: Datei {filename} nicht gefunden!")

In [ ]:
# ============================================================================
# BLOCK 3: MODEL ARCHITECTURES
# ============================================================================
def build_custom_cnn(filters_base=32, depth=3, dropout=0.5, dense_units=512):
    model = models.Sequential([layers.Input(shape=(28, 28, 3))])
    for i in range(depth):
        filters = filters_base * (2 ** i)
        model.add(layers.Conv2D(filters, (3, 3), activation='relu', padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Dropout(dropout * (0.5 + i*0.25)))

    model.add(layers.Flatten())
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(2, activation='softmax'))
    return model

def build_transfer_model(base_name='resnet50', dropout=0.5, unfreeze_layers=None, dense_units=256):
    inputs = layers.Input(shape=(28, 28, 3))
    # Upsampling is crucial for pre-trained models
    x = layers.UpSampling2D(size=(2, 2), interpolation='bilinear')(inputs)

    base_models = {
        'resnet50': lambda: ResNet50(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg'),
        'vgg16': lambda: VGG16(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg'),
        'efficientnet': lambda: EfficientNetB0(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg')
    }

    if base_name not in base_models: raise ValueError(f"Unknown base: {base_name}")
    base = base_models[base_name]()

    # Fine-tuning logic
    if unfreeze_layers is None:
        base.trainable = True # Full fine-tuning
    elif unfreeze_layers == 0:
        base.trainable = False # Feature extraction
    else:
        base.trainable = True
        for layer in base.layers[:-unfreeze_layers]:
            layer.trainable = False

    x = base(x, training=False)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout * 0.5)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return models.Model(inputs, outputs, name=f"{base_name}_model")

In [ ]:
# ============================================================================
# BLOCK 4: TRAINING CONFIGURATION & HELPERS
# ============================================================================
def compile_model(model, learning_rate):
    """
    Compile with only AUC metric
    Precision/Recall per-batch are unreliable with severe class imbalance
    """
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc')  # Only stable metric during training
        ]
    )


def compile_model(model, learning_rate):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(class_id=1, name='precision'),  # ← class_id=1 = Malignant
            tf.keras.metrics.Recall(class_id=1, name='recall')         # ← class_id=1 = Malignant
        ]
    )



def get_callbacks(config, model_path):
    return [
        callbacks.EarlyStopping(
            monitor='val_recall',  # ← Changed from 'val_auc'
            patience=config['patience'],
            mode='max',
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=model_path,
            monitor='val_recall',  # ← Changed from 'val_auc'
            mode='max',
            save_best_only=True,
            verbose=0
        ),
        WandbMetricsLogger(log_freq='epoch'),
        WandbModelCheckpoint(model_path, monitor='val_recall', mode='max', save_best_only=True)
    ]

def compute_metrics(model, data_dict, split_name):
    """Only compute essential metrics on full dataset"""
    y_pred_probs = model.predict(data_dict['x'], verbose=0)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(data_dict['y'], axis=1)

    metrics = {
        f'{split_name}/auc': roc_auc_score(data_dict['y'], y_pred_probs),
        f'{split_name}/recall_mal': recall_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0),
        f'{split_name}/precision_mal': precision_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0),
        f'{split_name}/f1_mal': f1_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0)
    }

    return metrics, confusion_matrix(y_true_classes, y_pred_classes)


In [ ]:
# ============================================================================
# BLOCK 5: MAIN TRAINING LOOP (CRASH-PROOF VERSION)
# ============================================================================
def train_experiment(config, data, datagen):
    print(f"\n{'='*60}")
    print(f"🚀 STARTING: {config['name']}")
    print(f"{'='*60}")

    model = None
    run = None

    try:
        # 1. Init W&B
        if wandb.run is not None: wandb.finish()
        run = wandb.init(project="DermaMNIST_Study", name=config['name'], config=config, reinit=True, id=wandb.util.generate_id())

        # 2. Build Model
        if config['architecture'] == 'custom_cnn':
            model = build_custom_cnn(config['filters_base'], config['depth'], config['dropout'], config['dense_units'])
        else:
            model = build_transfer_model(config['architecture'], config['dropout'], config['unfreeze_layers'], config['dense_units'])

        compile_model(model, config['learning_rate'])
        model_path = f"{config['name']}.keras"

        # 3. CRASH-PROOF DATASET GENERATION
        # Wir erstellen den Generator explizit
        train_gen = datagen.flow(data['x_train'], data['y_train'], batch_size=config['batch_size'], seed=SEED)

        # Sicherheits-Berechnung der Schritte
        steps = len(data['x_train']) // config['batch_size']

        # TRICK: Wir wandeln den Generator in ein tf.data.Dataset um und nutzen .repeat()
        # Das garantiert, dass ihm NIEMALS die Daten ausgehen.
        train_dataset = tf.data.Dataset.from_generator(
            lambda: train_gen,
            output_signature=(
                tf.TensorSpec(shape=(None, 28, 28, 3), dtype=tf.float32),
                tf.TensorSpec(shape=(None, 2), dtype=tf.float32)
            )
        ).repeat() # <--- DAS IST DER FIX: Unendliche Wiederholung

        # 4. Train
        history = model.fit(
            train_dataset, # Wir nutzen das unendliche Dataset
            steps_per_epoch=steps, # Hier ist -1 nicht mehr nötig, da .repeat() existiert
            epochs=config['epochs'],
            validation_data=(data['x_val'], data['y_val']),
            class_weight=data['class_weights'],
            callbacks=get_callbacks(config, model_path),
            verbose=1
        )

        # 5. Evaluation & Logging (wie gehabt)
        final_train_auc = history.history['auc'][-1]
        final_val_auc = history.history['val_auc'][-1]
        overfitting_gap_auc = final_train_auc - final_val_auc

        test_metrics, cm = compute_metrics(model, {'x': data['x_test'], 'y': data['y_test']}, 'test')

        test_metrics['overfitting_gap_auc'] = overfitting_gap_auc
        test_metrics['final_train_auc'] = final_train_auc
        test_metrics['final_val_auc'] = final_val_auc

        wandb.log(test_metrics)

        # Print Summary
        print(f"\n📊 RESULTS for {config['name']}:")
        print(f"   Val AUC: {final_val_auc:.4f}")
        print(f"   Test Recall: {test_metrics['test/recall_mal']:.4f}")
        print(f"   Test AUC: {test_metrics['test/auc']:.4f}")

        return {**test_metrics, 'config': config, 'history': history.history}

    except Exception as e:
        print(f"\n❌ ERROR: {e}")
        import traceback
        traceback.print_exc()
        return {'config': config, 'error': str(e)}

    finally:
        print(f"🧹 Cleanup...")
        if run is not None: wandb.finish()
        K.clear_session()
        if model is not None: del model
        gc.collect()

In [ ]:
# ============================================================================
# BLOCK 6a: Custom CNN (Trainieren & Speichern)
# ============================================================================
config_cnn = {
    'name': 'P1_CNN_Baseline',
    'architecture': 'custom_cnn',
    'filters_base': 32, 'depth': 3,
    'dropout': 0.5, 'dense_units': 512,
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 50,  # Increased from 30
    'patience': 10,  # Increased from 10
    'unfreeze_layers': None
}

# 1. Trainieren
res_cnn = train_experiment(config_cnn, data, datagen)

# 2. Sofort ins Drive sichern
save_to_drive(config_cnn['name'])


🚀 STARTING: P1_CNN_Baseline


wandb: Currently logged in as: abraham-gezehei (abraham-gezehei-fachhochschule-nordwestschweiz-fhnw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: WARNING When using `save_best_only`, ensure that the `filepath` argument contains formatting placeholders like `{epoch:02d}` or `{batch:02d}`. This ensures correct interpretation of the logged artifacts.


Epoch 1/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 15s 73ms/step - auc: 0.6834 - loss: 0.8988 - val_auc: 0.8653 - val_loss: 0.5295
Epoch 2/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - auc: 0.9262 - loss: 0.3565 - val_auc: 0.9013 - val_loss: 1.1888
Epoch 3/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - auc: 0.9514 - loss: 0.2848 - val_auc: 0.9012 - val_loss: 1.5056
Epoch 4/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - auc: 0.9570 - loss: 0.2724 - val_auc: 0.8878 - val_loss: 1.1666
Epoch 5/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - auc: 0.9652 - loss: 0.2385 - val_auc: 0.9393 - val_loss: 0.3670
Epoch 6/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - auc: 0.9701 - loss: 0.2207 - val_auc: 0.9713 - val_loss: 0.2208
Epoch 7/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - auc: 0.9735 - loss: 0.2084 - val_auc: 0.9758 - val_loss: 0.2106
Epoch 8/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - auc: 0.9710 - loss: 0.2171 - val_auc: 0.9796 - val_loss: 0.1975
Epoch 9/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms

epoch/auc,▁▆▇▇▇███████████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_auc,▁▃▃▂▅▇▇█▇█▇█▆█▇█▇▆▇█▇██▄▇█▆▅▇▇██
epoch/val_loss,▃▆█▆▂▁▁▁▁▁▁▁▂▁▂▁▁▂▂▁▂▁▁▄▂▁▂▃▂▂▂▁
final_train_auc,▁
final_val_auc,▁
overfitting_gap_auc,▁
test/auc,▁
+3,...


💾 GESPEICHERT: P1_CNN_Baseline -> Google Drive


In [ ]:
# ============================================================================
# BLOCK 6b: ResNet50 (Trainieren & Speichern)
# ============================================================================
config_resnet = {
    'name': 'P1_ResNet50_Baseline',
    'architecture': 'resnet50',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001,
    'batch_size': 32,
    'epochs': 50,  # Increased
    'patience': 15   # Increased
}

res_resnet = train_experiment(config_resnet, data, datagen)
save_to_drive(config_resnet['name'])


🚀 STARTING: P1_ResNet50_Baseline


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 85s 209ms/step - auc: 0.6088 - loss: 0.9773 - val_auc: 0.9013 - val_loss: 7.0501
Epoch 2/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 33s 149ms/step - auc: 0.8354 - loss: 0.6052 - val_auc: 0.8882 - val_loss: 0.5199
Epoch 3/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 31s 143ms/step - auc: 0.9055 - loss: 0.4233 - val_auc: 0.8931 - val_loss: 0.3529
Epoch 4/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 43s 195ms/step - auc: 0.9342 - loss: 0.3534 - val_auc: 0.9239 - val_loss: 0.3180
Epoch 5/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 42s 192ms/step - auc: 0.9533 - loss: 0.2911 - val_auc: 0.9699 - val_loss: 0.2688
Epoch 6/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 46s 210ms/step - auc: 0.9696 - loss: 0.2318 - val_auc: 0.9822 - val_loss: 0.2073
Epoch 7/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 31s 140ms/step - auc: 0.9678 - loss: 0.2306 - val_auc: 0.9766 - val_loss: 0.1922
Epoch 8/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 47s 218ms/step - auc: 0.9781 - loss: 0.1888 - val_auc: 0.9843 

epoch/auc,▁▅▆▇▇██████████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_auc,▂▁▁▄▇█▇███▇██████▄█████████▇███
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁
final_train_auc,▁
final_val_auc,▁
overfitting_gap_auc,▁
test/auc,▁
+3,...


💾 GESPEICHERT: P1_ResNet50_Baseline -> Google Drive


In [ ]:
# ============================================================================
# BLOCK 6c: VGG16 (Trainieren & Speichern)
# ============================================================================
config_vgg = {
    'name': 'P1_VGG16_Baseline',
    'architecture': 'vgg16',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001,
    'batch_size': 32,
    'epochs': 50,
    'patience': 15
}

res_vgg = train_experiment(config_vgg, data, datagen)
save_to_drive(config_vgg['name'])


🚀 STARTING: P1_VGG16_Baseline


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 27s 103ms/step - auc: 0.5374 - loss: 0.8988 - val_auc: 0.1580 - val_loss: 0.8796
Epoch 2/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - auc: 0.7589 - loss: 0.5798 - val_auc: 0.9649 - val_loss: 0.3110
Epoch 3/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 22s 103ms/step - auc: 0.8737 - loss: 0.4552 - val_auc: 0.9701 - val_loss: 0.4851
Epoch 4/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 28s 128ms/step - auc: 0.9187 - loss: 0.3805 - val_auc: 0.9767 - val_loss: 0.4218
Epoch 5/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 19s 85ms/step - auc: 0.9377 - loss: 0.3325 - val_auc: 0.9767 - val_loss: 0.2973
Epoch 6/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - auc: 0.9517 - loss: 0.2926 - val_auc: 0.9256 - val_loss: 0.3218
Epoch 7/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - auc: 0.9255 - loss: 0.3465 - val_auc: 0.9620 - val_loss: 0.2558
Epoch 8/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - auc: 0.9523 - loss: 0.2811 - val_auc: 0.9688 - va

epoch/auc,▁▄▆▇▇▇▇▇▇███████████████████████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_auc,▁███▇█████████████▇████████▇███▆██▇█████
epoch/val_loss,█▂▄▃▂▂▂▂▂▁▂▂▁▃▁▁▂▂▃▄▃▂▃▃▂▂▂▃▃▁▃▂▅▂▃▂▂▁▂▂
final_train_auc,▁
final_val_auc,▁
overfitting_gap_auc,▁
test/auc,▁
+3,...


💾 GESPEICHERT: P1_VGG16_Baseline -> Google Drive


In [ ]:
# ============================================================================
# BLOCK 6d: EfficientNet (Trainieren & Speichern)
# ============================================================================
config_effnet = {
    'name': 'P1_EfficientNet_Baseline',
    'architecture': 'efficientnet',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001,
    'batch_size': 32,
    'epochs': 50,
    'patience': 15
}

res_effnet = train_experiment(config_effnet, data, datagen)
save_to_drive(config_effnet['name'])


🚀 STARTING: P1_EfficientNet_Baseline


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 72s 142ms/step - auc: 0.5697 - loss: 1.0030 - val_auc: 0.8918 - val_loss: 0.5180
Epoch 2/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - auc: 0.7041 - loss: 0.7213 - val_auc: 0.7724 - val_loss: 0.6049
Epoch 3/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 26s 118ms/step - auc: 0.8206 - loss: 0.5363 - val_auc: 0.9155 - val_loss: 0.3779
Epoch 4/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 23s 105ms/step - auc: 0.8761 - loss: 0.4465 - val_auc: 0.9057 - val_loss: 0.4249
Epoch 5/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 114ms/step - auc: 0.9122 - loss: 0.3781 - val_auc: 0.5728 - val_loss: 0.6894
Epoch 6/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - auc: 0.9394 - loss: 0.3143 - val_auc: 0.8856 - val_loss: 0.4050
Epoch 7/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 23s 104ms/step - auc: 0.9493 - loss: 0.2860 - val_auc: 0.8663 - val_loss: 3.2545
Epoch 8/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - auc: 0.9609 - loss: 0.2509 - val_auc: 0.8942 

epoch/auc,▁▃▅▆▇███████████████████████████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_auc,▇▅▇▇▁▇▇▇▇▇▇▇▇█▇██▇▇▇▇▇██▇▇▇▇▇▇█▇▇▇█▇█▇█▇
epoch/val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁
final_train_auc,▁
final_val_auc,▁
overfitting_gap_auc,▁
test/auc,▁
+3,...


💾 GESPEICHERT: P1_EfficientNet_Baseline -> Google Drive


In [ ]:
# ============================================================================
# BLOCK 6e: COMPARE RESULTS (UPDATED)
# ============================================================================

# Collect all results
all_results = []

if 'res_cnn' in locals(): all_results.append(res_cnn)
if 'res_resnet' in locals(): all_results.append(res_resnet)
if 'res_vgg' in locals(): all_results.append(res_vgg)
if 'res_effnet' in locals(): all_results.append(res_effnet)

if all_results:
    # Create comparison dataframe
    comparison_data = []

    for res in all_results:
        # Skip if error occurred
        if 'error' in res:
            print(f"⚠️ Skipping {res['config']['name']} (failed during training)")
            continue

        comparison_data.append({
            'Architecture': res['config']['architecture'],
            'Name': res['config']['name'],
            'Test_Recall': f"{res['test/recall_mal']:.4f}",
            'Test_Precision': f"{res['test/precision_mal']:.4f}",
            'Test_F1': f"{res['test/f1_mal']:.4f}",
            'Test_AUC': f"{res['test/auc']:.4f}",
            'Overfitting_Gap': f"{res['overfitting_gap']:.4f}"
        })

    df_compare = pd.DataFrame(comparison_data)

    print("\n" + "="*80)
    print("🏆 PHASE 1 RESULTS SUMMARY")
    print("="*80 + "\n")
    print(df_compare.to_string(index=False))

    # Find best by Test AUC
    best_idx = df_compare['Test_AUC'].astype(float).idxmax()

    print("\n" + "="*80)
    print("🥇 BEST MODEL (by Test AUC):")
    print(f"   Architecture: {df_compare.loc[best_idx, 'Architecture']}")
    print(f"   Test Recall:  {df_compare.loc[best_idx, 'Test_Recall']} ← PRIMARY METRIC")
    print(f"   Test AUC:     {df_compare.loc[best_idx, 'Test_AUC']}")
    print(f"   Overfitting:  {df_compare.loc[best_idx, 'Overfitting_Gap']}")
    print("="*80 + "\n")

    # Save to CSV
    df_compare.to_csv('phase1_results.csv', index=False)
    print("💾 Results saved to: phase1_results.csv")

else:
    print("⚠️ No results available yet. Run blocks 6a-6d first.")

KeyError: 'overfitting_gap'

In [ ]:
# ============================================================================
# BLOCK 7: HYPERPARAMETER TUNING (Winner from Phase 1/2)
# ============================================================================

# 1. CONFIGURATION - Update based on your Phase 1/2 winner
WINNER_ARCH = 'resnet50'  # ← Change based on your results
USE_CLASS_WEIGHTS = True   # ← Set based on Phase 1 vs 2 comparison

# 2. Update class weights in data dict
if USE_CLASS_WEIGHTS:
    data['class_weights'] = {0: 1.0, 1: 3.0}
    weight_tag = 'Weighted'
else:
    data['class_weights'] = None
    weight_tag = 'NoWeight'

# 3. Define tuning configurations
tuning_configs = []

if WINNER_ARCH == 'custom_cnn':
    # CNN-specific tuning
    tuning_configs = [
        {
            'name': f'P3_CNN_Depth4_{weight_tag}',
            'architecture': 'custom_cnn',
            'filters_base': 32, 'depth': 4,  # ← Deeper
            'dropout': 0.5, 'dense_units': 512,
            'learning_rate': 0.001, 'batch_size': 64,
            'epochs': 30, 'patience': 10, 'unfreeze_layers': None
        },
        {
            'name': f'P3_CNN_MoreFilters_{weight_tag}',
            'architecture': 'custom_cnn',
            'filters_base': 64, 'depth': 3,  # ← More filters
            'dropout': 0.5, 'dense_units': 512,
            'learning_rate': 0.001, 'batch_size': 64,
            'epochs': 30, 'patience': 10, 'unfreeze_layers': None
        },
        {
            'name': f'P3_CNN_HighDropout_{weight_tag}',
            'architecture': 'custom_cnn',
            'filters_base': 32, 'depth': 3,
            'dropout': 0.7, 'dense_units': 512,  # ← Higher dropout
            'learning_rate': 0.001, 'batch_size': 64,
            'epochs': 30, 'patience': 10, 'unfreeze_layers': None
        },
    ]
else:
    # Transfer learning tuning (ResNet50, VGG16, EfficientNet)
    tuning_configs = [
        {
            'name': f'P3_{WINNER_ARCH}_Freeze10_{weight_tag}',
            'architecture': WINNER_ARCH,
            'dropout': 0.5, 'dense_units': 256,
            'unfreeze_layers': 10,  # ← Partial freeze
            'learning_rate': 0.0001, 'batch_size': 32,
            'epochs': 30, 'patience': 10
        },
        {
            'name': f'P3_{WINNER_ARCH}_Freeze20_{weight_tag}',
            'architecture': WINNER_ARCH,
            'dropout': 0.5, 'dense_units': 256,
            'unfreeze_layers': 20,  # ← More layers unfrozen
            'learning_rate': 0.0001, 'batch_size': 32,
            'epochs': 30, 'patience': 10
        },
        {
            'name': f'P3_{WINNER_ARCH}_HighDropout_{weight_tag}',
            'architecture': WINNER_ARCH,
            'dropout': 0.7, 'dense_units': 256,  # ← Higher dropout
            'unfreeze_layers': None,
            'learning_rate': 0.0001, 'batch_size': 32,
            'epochs': 30, 'patience': 10
        },
        {
            'name': f'P3_{WINNER_ARCH}_LowLR_{weight_tag}',
            'architecture': WINNER_ARCH,
            'dropout': 0.5, 'dense_units': 256,
            'unfreeze_layers': None,
            'learning_rate': 0.00005,  # ← Lower learning rate
            'batch_size': 32, 'epochs': 30, 'patience': 10
        },
    ]

# 4. Run tuning experiments
print(f"\n{'='*60}")
print(f"🔧 PHASE 3: TUNING {WINNER_ARCH.upper()}")
print(f"   Class Weights: {data['class_weights']}")
print(f"   Experiments: {len(tuning_configs)}")
print(f"{'='*60}\n")

phase3_results = []

for config in tuning_configs:
    # Train
    result = train_experiment(config, data, datagen)
    phase3_results.append(result)

    # Save to Drive immediately
    save_to_drive(config['name'])

# 5. Compare tuning results
print(f"\n{'='*60}")
print(f"📊 PHASE 3 TUNING RESULTS")
print(f"{'='*60}\n")

tuning_comparison = []
for res in phase3_results:
    if 'error' not in res:
        tuning_comparison.append({
            'Name': res['config']['name'],
            'Val_Recall': f"{res.get('final_val_recall', 0):.4f}",
            'Test_Recall': f"{res['test/recall_mal']:.4f}",
            'Test_Precision': f"{res['test/precision_mal']:.4f}",
            'Test_AUC': f"{res['test/auc']:.4f}",
            'Overfitting': f"{res.get('overfitting_gap', 0):.4f}"
        })

df_tuning = pd.DataFrame(tuning_comparison)
print(df_tuning.to_string(index=False))

# 6. Identify best model
best_idx = df_tuning['Val_Recall'].astype(float).idxmax()
BEST_MODEL_NAME = df_tuning.loc[best_idx, 'Name']

print(f"\n🏆 BEST TUNED MODEL: {BEST_MODEL_NAME}")
print(f"   Val Recall: {df_tuning.loc[best_idx, 'Val_Recall']}")
print(f"\n💾 All models saved to: {SAVE_DIR}")

In [ ]:
# ============================================================================
# BLOCK 8: FINAL TEST EVALUATION (RUN ONLY ONCE!)
# ============================================================================

# ⚠️ WARNING: Only run this block ONCE after all tuning is complete!
# This is your final, reportable result.

# 1. Specify best model from Phase 3
FINAL_MODEL_NAME = 'P3_resnet50_Freeze20_Weighted'  # ← Update with your winner

# 2. Load model from Drive
final_model_path = os.path.join(SAVE_DIR, f'{FINAL_MODEL_NAME}.keras')

print(f"{'='*60}")
print(f"🏁 FINAL TEST EVALUATION")
print(f"{'='*60}")
print(f"Loading: {final_model_path}")

try:
    final_model = tf.keras.models.load_model(final_model_path)
    print("✅ Model loaded successfully\n")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# 3. Evaluate on TEST SET (first and only time!)
print("📊 Evaluating on TEST SET...")

test_metrics, test_cm = compute_metrics(
    final_model,
    {'x': data['x_test'], 'y': data['y_test']},
    'final_test'
)

# 4. Print Final Results
print(f"\n{'='*60}")
print(f"🏆 FINAL RESULTS - {FINAL_MODEL_NAME}")
print(f"{'='*60}")
print(f"\n📈 PRIMARY METRIC:")
print(f"   Recall (Sensitivity): {test_metrics['final_test/recall_mal']:.4f}")

print(f"\n📊 SECONDARY METRICS:")
print(f"   Precision:            {test_metrics['final_test/precision_mal']:.4f}")
print(f"   F1 Score:             {test_metrics['final_test/f1_mal']:.4f}")
print(f"   AUC:                  {test_metrics['final_test/auc']:.4f}")

print(f"\n🎯 CONFUSION MATRIX:")
print(f"                    Predicted")
print(f"                  Benign  Malignant")
print(f"   Actual Benign    {test_cm[0,0]:5d}    {test_cm[0,1]:5d}")
print(f"          Malig.    {test_cm[1,0]:5d}    {test_cm[1,1]:5d}")

# 5. Clinical Interpretation
total_malignant = test_cm[1,0] + test_cm[1,1]
detected = test_cm[1,1]
missed = test_cm[1,0]

print(f"\n🏥 CLINICAL INTERPRETATION:")
print(f"   Total malignant cases: {total_malignant}")
print(f"   Correctly detected:    {detected} ({detected/total_malignant*100:.1f}%)")
print(f"   Missed (False Neg):    {missed} ({missed/total_malignant*100:.1f}%)")

print(f"\n{'='*60}")
print(f"✅ EVALUATION COMPLETE")
print(f"{'='*60}")

# 6. Save final results to CSV
final_results = {
    'model_name': FINAL_MODEL_NAME,
    'test_recall': test_metrics['final_test/recall_mal'],
    'test_precision': test_metrics['final_test/precision_mal'],
    'test_f1': test_metrics['final_test/f1_mal'],
    'test_auc': test_metrics['final_test/auc'],
    'true_neg': test_cm[0,0],
    'false_pos': test_cm[0,1],
    'false_neg': test_cm[1,0],
    'true_pos': test_cm[1,1]
}

df_final = pd.DataFrame([final_results])
final_csv_path = os.path.join(SAVE_DIR, 'FINAL_TEST_RESULTS.csv')
df_final.to_csv(final_csv_path, index=False)
print(f"\n💾 Results saved to: {final_csv_path}")